## **WhisperTranscribe — Audio Transcription Tool**

**WhisperTranscribe** uses OpenAI's Whisper speech recognition model (via the [faster-whisper](https://github.com/SYSTRAN/faster-whisper) engine) to transcribe audio files.

**Supported audio formats:** mp3, wav, m4a, flac, ogg, wma, aac, and others supported by ffmpeg.

**Output:** plain text (.txt), markdown (.md), SRT subtitles (.srt, optional), audio with silence removed (.mp3, optional).

**Languages:** 99 languages with automatic detection.

### **How to use**

1. **Section 1** — Run cells 1.1-1.2 to install dependencies and check your environment
2. **Section 2** — Select a preset and configure options
3. **Section 3** — Upload your audio file(s) (one or multiple)
4. **Section 4** — Run transcription (all files processed automatically)
5. **Section 5** — Download a single ZIP archive with all results

### **Runtime recommendations**

Go to **Runtime → Change runtime type → GPU** before running.

| Preset | Whisper model | Min VRAM | Recommended runtime |
|--------|--------------|----------|--------------------|
| **small** | `small` | ~1 GB | Any GPU (T4, V100, A100) |
| **medium** | `medium` | ~2.5 GB | T4 or better |
| **high** | `large-v3` | ~5 GB | T4 (15 GB), V100, A100 |

All presets work on the free Colab T4 GPU. Without GPU, transcription runs on CPU (much slower).

In [ ]:
#@title **1.1 Install dependencies**
!pip install -q faster-whisper pydub

print("Dependencies installed successfully.")

In [ ]:
#@title **1.2 Environment check**
import torch
import shutil
import os

print("=" * 50)
print("ENVIRONMENT")
print("=" * 50)

# --- GPU ---
gpu_available = torch.cuda.is_available()
if gpu_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_mem_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    print(f"GPU:        {gpu_name}")
    print(f"VRAM:       {gpu_mem_free:.1f} / {gpu_mem_total:.1f} GB free")
    DEVICE = "cuda"
    COMPUTE_TYPE = "float16"
else:
    print("GPU:        NOT DETECTED")
    print("WARNING:    Go to Runtime > Change runtime type > GPU (T4)")
    gpu_mem_total = 0
    DEVICE = "cpu"
    COMPUTE_TYPE = "int8"

# --- RAM ---
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"RAM:        {ram.available / 1e9:.1f} / {ram.total / 1e9:.1f} GB free")
except ImportError:
    mem = dict(line.split(":", 1) for line in open("/proc/meminfo") if "MemTotal" in line or "MemAvailable" in line)
    total = int(mem.get("MemTotal", "0 kB").strip().split()[0]) / 1e6
    avail = int(mem.get("MemAvailable", "0 kB").strip().split()[0]) / 1e6
    print(f"RAM:        {avail:.1f} / {total:.1f} GB free")

# --- Disk ---
stat = os.statvfs("/")
disk_total = stat.f_blocks * stat.f_frsize / 1e9
disk_free = stat.f_bavail * stat.f_frsize / 1e9
print(f"Disk:       {disk_free:.1f} / {disk_total:.1f} GB free")

# --- ffmpeg ---
ffmpeg_path = shutil.which("ffmpeg")
if ffmpeg_path:
    print("ffmpeg:     available")
else:
    print("ffmpeg:     NOT FOUND — run: !apt install ffmpeg")

# --- Recommendation ---
print("=" * 50)
if gpu_mem_total >= 10:
    recommended = "high"
elif gpu_mem_total >= 4:
    recommended = "medium"
elif gpu_mem_total >= 1:
    recommended = "small"
else:
    recommended = "small (CPU mode — will be slow)"
print(f"Recommended preset: {recommended}")
print(f"Device: {DEVICE}, Compute type: {COMPUTE_TYPE}")

In [ ]:
#@title **2.1 Configure transcription**

#@markdown **Preset** — controls model size, speed, and quality.
preset = "medium" #@param ["small", "medium", "high"]

#@markdown **Language** — `auto` for automatic detection, or select manually.
language = "auto" #@param ["auto", "en", "es", "fr", "de", "it", "pt", "nl", "ru", "zh", "ja", "ko", "ar", "hi", "pl", "uk", "tr", "cs", "sv", "da", "fi", "el", "he", "th", "vi", "id", "ms", "ro", "hu", "bg", "hr", "sk", "sl", "sr", "lt", "lv", "et"]

#@markdown **Generate SRT subtitles** (with timestamps)
generate_srt = True #@param {type:"boolean"}

#@markdown **Export audio with silence removed** (uses VAD segments)
export_no_silence = False #@param {type:"boolean"}

# --- Preset definitions ---
PRESETS = {
    "small":  {"model_size": "small",    "beam_size": 1},
    "medium": {"model_size": "medium",   "beam_size": 3},
    "high":   {"model_size": "large-v3", "beam_size": 5},
}

config = PRESETS[preset]
MODEL_SIZE = config["model_size"]
BEAM_SIZE = config["beam_size"]
LANGUAGE = None if language == "auto" else language

print(f"Preset:      {preset}")
print(f"  Model:     {MODEL_SIZE}")
print(f"  Beam size: {BEAM_SIZE}")
print(f"  Language:  {'auto-detect' if LANGUAGE is None else LANGUAGE}")
print(f"  SRT:       {generate_srt}")
print(f"  No-silence audio: {export_no_silence}")
print(f"\nSettings configured.")

In [ ]:
#@title **3.1 Upload audio file(s)**
#@markdown Upload one or more audio files (mp3, wav, m4a, flac, ogg, aac, wma, etc.)

from google.colab import files
import os

print("Select your audio file(s)...")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No files uploaded. Run this cell again.")

AUDIO_FILES = []
for filename in uploaded.keys():
    path = os.path.join("/content", filename)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    AUDIO_FILES.append({"filename": filename, "path": path})
    print(f"  {filename} ({size_mb:.1f} MB)")

print(f"\nTotal files: {len(AUDIO_FILES)}")

In [ ]:
#@title **4.1 Run transcription**
#@markdown Transcribes all uploaded files. May take a while for multiple/long files.

from faster_whisper import WhisperModel
import time
import shutil

# --- Prepare output directory ---
OUTPUT_DIR = "/content/transcripts"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

# --- Load model once ---
print(f"Loading model '{MODEL_SIZE}' on {DEVICE}...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print("Model loaded.\n")

def format_timestamp(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"

def format_srt_time(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

# --- Process each file ---
total_start = time.time()

for file_idx, audio_file in enumerate(AUDIO_FILES, 1):
    filename = audio_file["filename"]
    audio_path = audio_file["path"]
    base_name = os.path.splitext(filename)[0]

    print(f"\n{'=' * 50}")
    print(f"[{file_idx}/{len(AUDIO_FILES)}] {filename}")
    print(f"{'=' * 50}")

    start_time = time.time()

    segments_raw, info = model.transcribe(
        audio_path,
        beam_size=BEAM_SIZE,
        language=LANGUAGE,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
    )

    segments = []
    for seg in segments_raw:
        segments.append(seg)
        print(f"  [{seg.start:7.1f}s -> {seg.end:7.1f}s]  {seg.text.strip()}")

    elapsed = time.time() - start_time
    audio_duration = info.duration

    print(f"\n  Language:        {info.language} ({info.language_probability:.0%})")
    print(f"  Audio duration:  {audio_duration:.0f}s ({audio_duration/60:.1f} min)")
    print(f"  Processing time: {elapsed:.1f}s ({audio_duration/elapsed:.1f}x realtime)")

    # --- Save TXT ---
    full_text = "\n".join(seg.text.strip() for seg in segments)
    txt_path = os.path.join(OUTPUT_DIR, f"{base_name}_transcript.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(full_text)
    print(f"  Saved: {txt_path}")

    # --- Save MD ---
    md_lines = []
    md_lines.append(f"# Transcript: {filename}\n")
    md_lines.append(f"- **Language:** {info.language} ({info.language_probability:.0%})")
    md_lines.append(f"- **Duration:** {format_timestamp(audio_duration)}")
    md_lines.append(f"- **Model:** {MODEL_SIZE} (preset: {preset})")
    md_lines.append(f"\n---\n")
    for seg in segments:
        ts = format_timestamp(seg.start)
        md_lines.append(f"**[{ts}]** {seg.text.strip()}\n")

    md_path = os.path.join(OUTPUT_DIR, f"{base_name}_transcript.md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))
    print(f"  Saved: {md_path}")

    # --- Save SRT ---
    if generate_srt:
        srt_path = os.path.join(OUTPUT_DIR, f"{base_name}_transcript.srt")
        with open(srt_path, "w", encoding="utf-8") as f:
            for i, seg in enumerate(segments, 1):
                f.write(f"{i}\n")
                f.write(f"{format_srt_time(seg.start)} --> {format_srt_time(seg.end)}\n")
                f.write(f"{seg.text.strip()}\n\n")
        print(f"  Saved: {srt_path}")

    # --- Export audio without silence ---
    if export_no_silence:
        from pydub import AudioSegment

        audio = AudioSegment.from_file(audio_path)
        chunks = [audio[int(seg.start * 1000):int(seg.end * 1000)] for seg in segments]

        if chunks:
            result = chunks[0]
            for chunk in chunks[1:]:
                result += chunk
            no_silence_path = os.path.join(OUTPUT_DIR, f"{base_name}_no_silence.mp3")
            result.export(no_silence_path, format="mp3")
            original_dur = len(audio) / 1000
            trimmed_dur = len(result) / 1000
            print(f"  Saved: {no_silence_path}")
            print(f"    {original_dur:.0f}s -> {trimmed_dur:.0f}s (removed {original_dur - trimmed_dur:.0f}s of silence)")

    # --- Preview ---
    print(f"\n  Preview: {full_text[:200]}{'...' if len(full_text) > 200 else ''}")

total_elapsed = time.time() - total_start
print(f"\n{'=' * 50}")
print(f"ALL DONE — {len(AUDIO_FILES)} file(s) in {total_elapsed:.1f}s")
print(f"Output: {OUTPUT_DIR}/")
print(f"{'=' * 50}")

In [ ]:
#@title **5.1 Download results**
#@markdown Archives all transcripts into a single ZIP and downloads it.

from google.colab import files
from datetime import datetime
import zipfile
import os

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_name = f"transcripts_{timestamp}.zip"
zip_path = f"/content/{zip_name}"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(OUTPUT_DIR):
        for fname in sorted(filenames):
            full_path = os.path.join(root, fname)
            arcname = fname
            zf.write(full_path, arcname)
            print(f"  + {fname}")

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\nArchive: {zip_name} ({zip_size_mb:.1f} MB)")

files.download(zip_path)
print("Download started.")